-------------------------------------
# SETUP
-------------------------------------

## Step 1 — Parse the PDFs
This step extracts text from the research papers using PyMuPDF while handling two-column layouts, headers/footers, page breaks, and common PDF formatting issues.

The extracted text is cleaned and saved with `[[PAGE n]]` markers so that page-level citations can be preserved during retrieval and generation.

In [ ]:
!pip -q install pymupdf
try:
    import pymupdf as fitz
except ImportError:
    import fitz
import re, os, glob, unicodedata

os.makedirs("papers", exist_ok=True)
os.makedirs("papers_txt", exist_ok=True)
if not glob.glob("papers/*.pdf"):          # skips the upload if PDFs are already there
    from google.colab import files
    for name, data in files.upload().items():
        open(f"papers/{name}", "wb").write(data)

SKIP_PAGES = {  # 1-indexed: cover / declaration pages
    "AGeneticNeuro-FuzzySystemforDiagnosingClinical.pdf": {1},
    "Predicting_depression_level_based.pdf": {1, 2, 19},
}

def clean(text):
    text = unicodedata.normalize("NFKC", text)                        # ﬁ -> fi
    text = re.sub(r"[\uac00-\ud7a3]", "", text)                       # Hangul junk from math glyphs
    text = re.sub(r"[ \t]+\n", "\n", text)                            # strip trailing spaces FIRST
    text = re.sub(r"(?<=[a-z])-\n(?=[a-z])", "", text)                # then join hyphenated breaks
    text = re.sub(r"(?m)^\s*Journal Pre-proof\s*$\n?", "", text)      # mid-page watermark
    text = re.sub(r"(?m)^https://doi\.org/10\.1371/\S+\s*$\n?", "", text)  # PLOS figure/table DOI lines
    return text

def get_lines(page):
    H = page.rect.height
    lines = []
    for b in page.get_text("dict")["blocks"]:
        if b["type"] != 0:
            continue
        for l in b["lines"]:
            txt = "".join(s["text"] for s in l["spans"]).strip()
            x0, y0, x1, y1 = l["bbox"]
            if txt and y0 > 0.07 * H and y1 < 0.93 * H:      # drop running headers/footers
                lines.append((x0, y0, x1, y1, txt))
    return lines

def page_text(page):
    W, H = page.rect.width, page.rect.height
    mid = W / 2
    lines = get_lines(page)
    narrow = [l for l in lines if (l[2] - l[0]) < 0.55 * W]
    L = sum(1 for l in narrow if (l[0] + l[2]) / 2 < mid)
    R = len(narrow) - L

    if not (L >= 10 and R >= 10):                            # single column: keep block order
        blocks = [b for b in page.get_text("blocks")
                  if b[6] == 0 and b[4].strip() and b[1] > 0.07 * H and b[3] < 0.93 * H]
        blocks.sort(key=lambda b: b[1])
        return "\n".join(b[4].strip() for b in blocks), False

    lines.sort(key=lambda l: l[1])                           # two columns: line by line
    out, band = [], []
    def flush():
        out.extend(sorted([l for l in band if (l[0] + l[2]) / 2 < mid], key=lambda l: l[1]))
        out.extend(sorted([l for l in band if (l[0] + l[2]) / 2 >= mid], key=lambda l: l[1]))
        band.clear()
    for l in lines:
        if l[0] < mid < l[2] and (l[2] - l[0]) > 0.6 * W:    # full-width line
            flush(); out.append(l)
        else:
            band.append(l)
    flush()
    return "\n".join(l[4] for l in out), True

for path in sorted(glob.glob("papers/*.pdf")):
    base = os.path.basename(path)
    doc = fitz.open(path)
    pages = []
    for i, p in enumerate(doc, 1):
        if i in SKIP_PAGES.get(base, set()):
            continue
        t, _ = page_text(p)
        t = clean(t)
        if len(t) > 200:                                     # drops figure-only pages
            pages.append((i, t))

    open(f"papers_txt/{base}.txt", "w").write(
        "\n\n".join(f"[[PAGE {n}]]\n{t}" for n, t in pages))
    body = "\n".join(t for _, t in pages)
    print(f"{base[:45]:45} kept={len(pages)}/{len(doc)} chars={len(body)} "
          f"hyphens_left={len(re.findall(r'\w-\n\w', body))} "
          f"ligatures={sum(body.count(c) for c in 'ﬁﬂﬀﬃﬄ')} "
          f"watermark={body.count('Journal Pre-proof')}")
    if base.startswith("AGenetic"):
        d = dict(pages)
        print("   Genetic p4 column order ok:",
              d[4].find("Data collection") < d[4].find("represents the diagnostic"))


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.8/25.8 MB 31.7 MB/s eta 0:00:00


Saving A_machine_learning_based_depression_screening_fram.pdf to A_machine_learning_based_depression_screening_fram.pdf
Saving A_neuro-fuzzy_approach.pdf to A_neuro-fuzzy_approach.pdf
Saving AGeneticNeuro-FuzzySystemforDiagnosingClinical.pdf to AGeneticNeuro-FuzzySystemforDiagnosingClinical.pdf
Saving An_in-depth_analysis_of_machine.pdf to An_in-depth_analysis_of_machine.pdf
Saving Predicting_depression_level_based.pdf to Predicting_depression_level_based.pdf
AGeneticNeuro-FuzzySystemforDiagnosingClinica kept=5/8 chars=19912 hyphens_left=2 ligatures=0 watermark=0
   Genetic p4 column order ok: True
A_machine_learning_based_depression_screening kept=28/29 chars=89080 hyphens_left=5 ligatures=0 watermark=0
A_neuro-fuzzy_approach.pdf                    kept=9/9 chars=38571 hyphens_left=13 ligatures=0 watermark=0
An_in-depth_analysis_of_machine.pdf           kept=12/12 chars=62499 hyphens_left=11 ligatures=0 watermark=0
Predicting_depression_level_based.pdf         kept=16/19 chars=64208 h

## Step 2 — Chunk + Tag with Metadata

This step splits each research paper into smaller chunks using `RecursiveCharacterTextSplitter` while keeping the chunks within their respective sections.

Each chunk is tagged with metadata such as `paper`, `section`, `page`, `table_like`, and `chunk_id`, and the processed chunks are saved to `chunks.json` for later retrieval.

In [ ]:
!pip -q install langchain-text-splitters langchain-core

import re, os, glob, json, bisect, collections
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

PAPERS = {  # file stem -> short label used in citations (edit freely)
    "AGeneticNeuro-FuzzySystemforDiagnosingClinical": "Adegboye2021 (Genetic Neuro-Fuzzy)",
    "A_machine_learning_based_depression_screening_fram": "Khan2024 (EEG temporal features + ML)",
    "A_neuro-fuzzy_approach": "Chattopadhyay2017 (Neuro-fuzzy diagnosis)",
    "An_in-depth_analysis_of_machine": "Zulfiker2021 (ML + feature selection)",
    "Predicting_depression_level_based": "Saha2024 (Fuzzy logic depression level)",
}

SECTION_RULES = [   # checked in order; matched against the start of a short heading line
    ("back_matter", ("acknowledg", "author contributions", "supporting information", "data availability",
                     "code availability", "availability of data", "funding", "declaration of",
                     "competing interest", "conflict of interest")),
    ("abstract",     ("abstract",)),
    ("introduction", ("introduction",)),
    ("related_work", ("related work", "review of related", "background", "literature", "current literature")),
    ("methods",      ("method", "materials and method", "proposed")),
    ("results",      ("result", "experimental result", "experiment", "evaluation", "system testing")),
    ("discussion",   ("discussion", "findings", "implications", "theoretical", "managerial")),
    ("conclusion",   ("conclusion", "limitation", "future work", "future scope")),
]

def classify(line):
    """Return canonical section name if `line` looks like a section heading, else None."""
    s = line.strip()
    norm = re.sub(r"^\s*\d+(\.\d+)*\.?\s*", "", s.lower())
    if re.sub(r"\s+", "", norm).startswith("abstract"):                 # also "a b s t r a c t"
        return "abstract"
    if norm.startswith("future work") and ":" in norm[:14]:            # inline "Future work: ..."
        return "conclusion"
    body = re.sub(r"^\s*\d+(\.\d+)*\.?\s*", "", s)
    if not body or not body[0].isupper():                              # headings start with a capital
        return None
    if len(s) > 70 or len(body.split()) > 8 or re.search(r"[\d(),]", body) or s.endswith((".", ";")):
        return None                                                    # body text / table cells, not headings
    for name, prefixes in SECTION_RULES:
        if norm.startswith(prefixes):
            return name
    return None

# Saha's first page has an odd 2-column layout: its abstract text lands after the intro.
ABSTRACT_SPAN = {"Predicting_depression_level_based": ("Millions of individuals die", "distinct degrees")}

splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150, add_start_index=True)
docs, coverage = [], collections.defaultdict(collections.Counter)

for path in sorted(glob.glob("papers_txt/*.txt")):
    stem = os.path.basename(path).replace(".pdf.txt", "")
    if stem not in PAPERS:
        continue
    label = PAPERS[stem]
    raw = open(path).read()
    parts = re.split(r"\[\[PAGE (\d+)\]\]\n", raw)[1:]
    lines = [(int(n), l.strip()) for n, txt in zip(parts[0::2], parts[1::2])
             for l in txt.split("\n") if l.strip()]

    # cut the reference list (last line that is just "References")
    ref_idx = [i for i, (_, l) in enumerate(lines) if re.fullmatch(r"(?i)references?", l)]
    if ref_idx:
        lines = lines[:ref_idx[-1]]

    # group consecutive lines into (section, text, page-offsets) runs
    runs, cur, resume = [], "front", None
    span = ABSTRACT_SPAN.get(stem)
    for page, line in lines:
        if span and resume is None and span[0] in line:
            resume, cur = cur, "abstract"
        sec = None if (span and resume is not None) else classify(line)
        if sec and not line.lower().lstrip("0123456789. ").startswith("abstract"):
            cur = sec                                   # pure heading line: switch section, don't index it
            continue
        if sec == "abstract":
            cur = "abstract"
        if not runs or runs[-1]["section"] != cur:
            runs.append({"section": cur, "text": "", "starts": [], "pages": []})
        r = runs[-1]
        r["starts"].append(len(r["text"])); r["pages"].append(page)
        r["text"] += line + " "
        if span and resume is not None and span[1] in line:
            cur, resume = resume, None

    for r in runs:
        if r["section"] in ("front", "back_matter"):
            continue
        for d in splitter.create_documents([r["text"]], [{}]):
            text = d.page_content.strip()
            if len(text) < 150:
                continue
            page = r["pages"][bisect.bisect_right(r["starts"], d.metadata["start_index"]) - 1]
            digit_ratio = sum(c.isdigit() for c in text) / len(text)
            docs.append(Document(page_content=text, metadata={
                "paper": label, "section": r["section"], "page": page,
                "table_like": digit_ratio > 0.25}))
            coverage[label][r["section"]] += 1

for i, d in enumerate(docs):
    d.metadata["chunk_id"] = i

print(f"total chunks: {len(docs)}\n")
need = ["abstract", "introduction", "methods", "results", "conclusion"]
for label, cnt in coverage.items():
    missing = [s for s in need if cnt[s] == 0]
    print(f"{label}\n   {dict(cnt)}" + (f"\n   MISSING: {missing}" if missing else ""))

sample = next(d for d in docs if d.metadata["section"] == "conclusion")
print("\n--- sample conclusion chunk ---")
print(sample.metadata); print(sample.page_content[:500])

json.dump([{"text": d.page_content, "metadata": d.metadata} for d in docs],
          open("chunks.json", "w"), indent=1)


total chunks: 270

Adegboye2021 (Genetic Neuro-Fuzzy)
   {'abstract': 3, 'introduction': 5, 'related_work': 3, 'methods': 4, 'results': 2, 'discussion': 2, 'conclusion': 1}
Khan2024 (EEG temporal features + ML)
   {'abstract': 3, 'introduction': 2, 'related_work': 8, 'methods': 28, 'results': 31, 'conclusion': 2}
Chattopadhyay2017 (Neuro-fuzzy diagnosis)
   {'abstract': 2, 'introduction': 7, 'related_work': 7, 'methods': 11, 'results': 10, 'discussion': 3, 'conclusion': 1}
Zulfiker2021 (ML + feature selection)
   {'abstract': 2, 'introduction': 7, 'related_work': 8, 'methods': 29, 'results': 18, 'conclusion': 3}
Saha2024 (Fuzzy logic depression level)
   {'introduction': 7, 'abstract': 2, 'related_work': 9, 'methods': 27, 'results': 9, 'discussion': 11, 'conclusion': 3}

--- sample conclusion chunk ---
{'paper': 'Adegboye2021 (Genetic Neuro-Fuzzy)', 'section': 'conclusion', 'page': 7, 'table_like': False, 'chunk_id': 19}
As can be seen, a large amount of work has already been done in t

## Step 3 — Embeddings + FAISS

This step converts every chunk into vector embeddings using the local `sentence-transformers/all-MiniLM-L6-v2` model and stores them in a FAISS index.

A similarity search is also performed as a sanity check to verify that the index can retrieve relevant chunks before building the RAG pipeline on top of it.

In [ ]:
!pip -q install langchain-huggingface langchain-community faiss-cpu sentence-transformers

import json
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

# --- 1. reload chunks from chunks.json ---
raw = json.load(open("chunks.json"))
docs = [Document(page_content=r["text"], metadata=r["metadata"]) for r in raw]
print(f"loaded {len(docs)} chunks")

# --- 2. embeddings: free, local, no API key ---
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

# --- 3. build + save the FAISS index ---
vectorstore = FAISS.from_documents(docs, embeddings)
vectorstore.save_local("faiss_index")
print("FAISS index built and saved to faiss_index/")

# --- 4. sanity check: is retrieval even sensible? ---
query = "limitations of the proposed model"
hits = vectorstore.similarity_search(query, k=5)
for h in hits:
    print(f"[{h.metadata['paper']} | {h.metadata['section']} | p{h.metadata['page']}] {h.page_content[:120]}...")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 33.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 38.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 20.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 2.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


/tmp/ipykernel_816/2336215542.py:6: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


loaded 270 chunks


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

FAISS index built and saved to faiss_index/
[Saha2024 (Fuzzy logic depression level) | discussion | p15] unavailable. The model’s ability to perform well on unseen data, as shown in Eq. (11), is a crucial indicator of its gen...
[Saha2024 (Fuzzy logic depression level) | conclusion | p16] level could be a future project. In addition, further research, including a more varied sample representing many populat...
[Saha2024 (Fuzzy logic depression level) | discussion | p15] the sample size and including individuals from diverse and inclusive demographic backgrounds. • This study enriches the ...
[Zulfiker2021 (ML + feature selection) | introduction | p1] psychological analysis and psy- E-mail addresses: sabab.rumc@gmail.com (Md.S. Zulfiker), etykabir16@gmail.com (N. Kabir)...
[Adegboye2021 (Genetic Neuro-Fuzzy) | discussion | p7] the model. 5) The predictive power of ANFIS for diagnosing clinical depression was greatly enhanced by the inclusion of ...


# QUICK START

In [1]:
#unload the saved chunks, papers, eval
!unzip -o ResearchGapAnalyzer_backup.zip -d /

Archive:  ResearchGapAnalyzer_backup.zip
   creating: /content/papers/
  inflating: /content/papers/A_neuro-fuzzy_approach.pdf  
  inflating: /content/papers/Predicting_depression_level_based.pdf  
  inflating: /content/papers/AGeneticNeuro-FuzzySystemforDiagnosingClinical.pdf  
  inflating: /content/papers/An_in-depth_analysis_of_machine.pdf  
  inflating: /content/papers/A_machine_learning_based_depression_screening_fram.pdf  
   creating: /content/papers_txt/
  inflating: /content/papers_txt/A_neuro-fuzzy_approach.pdf.txt  
  inflating: /content/papers_txt/A_machine_learning_based_depression_screening_fram.pdf.txt  
  inflating: /content/papers_txt/AGeneticNeuro-FuzzySystemforDiagnosingClinical.pdf.txt  
  inflating: /content/papers_txt/An_in-depth_analysis_of_machine.pdf.txt  
  inflating: /content/papers_txt/Predicting_depression_level_based.pdf.txt  
   creating: /content/faiss_index/
  inflating: /content/faiss_index/index.pkl  
  inflating: /content/faiss_index/index.faiss  
  

-------------------------------------
# SINGLE-PAPER Q&A
-------------------------------------

## Step 4 — Per-Paper Retrieval

This step retrieves the most relevant chunks from each paper separately so that one paper cannot dominate the retrieved evidence.

The search now covers the whole paper by default instead of restricting retrieval to specific sections. Section information is retained as metadata for citations, while semantic similarity determines which chunks are retrieved.

In [ ]:
PAPER_LABELS = sorted(set(d.metadata["paper"] for d in docs))
print(f"{len(PAPER_LABELS)} papers:", *PAPER_LABELS, sep="\n  ")

def looks_tabular(text):
    """Catches flattened comparison-table rows that digit_ratio misses."""
    if text.count('%') >= 3:
        return True
    if '✔' in text or '✗' in text:
        return True
    if len(re.findall(r'\bNot applicable\b|\bNot mentioned\b|\bN/?A\b', text)) >= 2:
        return True
    return False

def per_paper_retrieve(query, k=3, sections=None, papers=None, fetch_k=None, exclude_table_like=True):
    """Retrieve top-k chunks from EACH paper separately.

    `sections=None` (the default) searches the WHOLE paper by meaning — section is metadata
    for citation, not a hard filter. Only narrow with `sections=[...]` when you deliberately
    want to restrict where a query looks (e.g. a known location, to save cost).
    `exclude_table_like=True` drops garbled/flattened table chunks; set False for questions
    whose answer might genuinely live in a table (e.g. a reported accuracy or component count).
    """
    papers = papers or PAPER_LABELS
    fetch_k = fetch_k or len(docs)   # search the WHOLE store before filtering, not LangChain's default top-20
    results = {}
    for paper in papers:
        def _filter(metadata, paper=paper):
            if metadata["paper"] != paper:
                return False
            if sections and metadata["section"] not in sections:
                return False
            if exclude_table_like and metadata.get("table_like"):
                return False
            return True
        # over-fetch, then drop tabular-looking text, then trim to k
        raw_hits = vectorstore.similarity_search(query, k=k + 3, filter=_filter, fetch_k=fetch_k)
        if exclude_table_like:
            raw_hits = [h for h in raw_hits if not looks_tabular(h.page_content)]
        results[paper] = raw_hits[:k]
    return results

# --- sanity check: whole-paper search, no section gate ---
import re
query = "limited to, shortcoming, drawback, future work, further research is needed"
per_paper = per_paper_retrieve(query, k=2)

for paper, hits in per_paper.items():
    print(f"\n=== {paper} ({len(hits)} hits) ===")
    if not hits:
        print("  (nothing matched)")
    for h in hits:
        print(f"  [{h.metadata['section']} p{h.metadata['page']}] {h.page_content[:150]}...")


In [ ]:
!pip -q install rank_bm25

import re
from rank_bm25 import BM25Okapi

PAPER_LABELS = sorted(set(d.metadata["paper"] for d in docs))
paper_docs = {p: [d for d in docs if d.metadata["paper"] == p] for p in PAPER_LABELS}

# --- lexical side: one BM25 index per paper (good at exact terms: "PCA", "92.03", "limited to") ---
_tok = lambda t: re.findall(r"[a-z0-9]+", t.lower())
bm25 = {p: BM25Okapi([_tok(d.page_content) for d in ds]) for p, ds in paper_docs.items()}

def looks_tabular(text):
    """Flags flattened table rows. NOTE: this also flags real answer text (e.g. Table 7's
    'Acc (%) 92.03'), so it is now OPT-IN via exclude_table_like=True, not the default."""
    if text.count('%') >= 3:
        return True
    if '✔' in text or '✗' in text:
        return True
    return len(re.findall(r'\bNot applicable\b|\bNot mentioned\b|\bN/?A\b', text)) >= 2

def per_paper_retrieve(query, k=3, sections=None, papers=None, exclude_table_like=False, rrf_k=60):
    """Hybrid retrieval, per paper: dense (FAISS/MiniLM) ranking + BM25 ranking, fused with
    Reciprocal Rank Fusion. `sections` / `exclude_table_like` are optional narrowing, off by default."""
    papers = papers or PAPER_LABELS
    results = {}
    for paper in papers:
        pdocs = paper_docs[paper]
        allowed = {d.metadata["chunk_id"] for d in pdocs
                   if (not sections or d.metadata["section"] in sections)
                   and not (exclude_table_like and (d.metadata.get("table_like") or looks_tabular(d.page_content)))}
        if not allowed:
            results[paper] = []
            continue

        # dense ranking over this paper's chunks (fetch_k = whole store, so the filter never starves)
        dense = vectorstore.similarity_search(
            query, k=len(pdocs), fetch_k=len(docs), filter=lambda m, p=paper: m["paper"] == p)
        dense_rank = {}
        for d in dense:
            if d.metadata["chunk_id"] in allowed:
                dense_rank[d.metadata["chunk_id"]] = len(dense_rank)

        # lexical ranking over the same chunks
        scores = bm25[paper].get_scores(_tok(query))
        bm25_rank = {}
        for i in sorted(range(len(pdocs)), key=lambda i: -scores[i]):
            cid = pdocs[i].metadata["chunk_id"]
            if cid in allowed:
                bm25_rank[cid] = len(bm25_rank)

        fused = {cid: 1 / (rrf_k + dense_rank.get(cid, len(pdocs))) + 1 / (rrf_k + bm25_rank[cid])
                 for cid in allowed}
        by_id = {d.metadata["chunk_id"]: d for d in pdocs}
        results[paper] = [by_id[c] for c in sorted(fused, key=fused.get, reverse=True)[:k]]
    return results

# ---------- FREE offline check: what does each eval question retrieve? (no Gemini calls) ----------
EVAL = [  # (key, question shown to the LLM, keyword-style search query, k)
    ("gap_limitations",  "What limitations are repeatedly mentioned across the papers?",
     "limitations shortcomings limited to drawback restricted future work further research", 3),
    ("gap_comparisons",  "Which approaches or methods are compared across the papers?",
     "compared with comparison outperforms baseline existing models classifiers versus", 3),
    ("gap_unresolved",   "What problems remain unresolved across the papers?",
     "remain unresolved challenges open problems not addressed future work further research needed", 3),
    ("gap_future_work",  "What future work do the authors propose across the papers?",
     "future work future research plan to extend further study will explore", 3),
    ("conflict_accuracy", "Which paper reports the highest accuracy?",
     "accuracy acc average accuracy percent achieved", 5),
    ("failure_pca_count", "How many papers use PCA?",
     "principal component analysis PCA dimensionality reduction", 3),
]

def preview(key):
    _, question, search_query, k = next(e for e in EVAL if e[0] == key)
    print("=" * 110); print(f"{key}\n  question    : {question}\n  search query: {search_query}")
    for paper, hits in per_paper_retrieve(search_query, k=k).items():
        print(f"\n  {paper}")
        for h in hits:
            print(f"    [{h.metadata['section']} p{h.metadata['page']}] {h.page_content[:130]}...")

for key, *_ in EVAL:
    preview(key)

gap_limitations
  question    : What limitations are repeatedly mentioned across the papers?
  search query: limitations shortcomings limited to drawback restricted future work further research

  Adegboye2021 (Genetic Neuro-Fuzzy)
    [abstract p2] work, a Genetic Neuro- Fuzzy Model which is capable of handling feature selection and extraction on depression dataset was propose...
    [related_work p3] In the previous study [14], a soft-computing model for depression prediction was presented. The objective of the research work is ...
    [discussion p7] 1) The efficiency of the system was significant when compared with the dataset obtained from Federal Medical Centre, Owo, Ondo Sta...

  Chattopadhyay2017 (Neuro-fuzzy diagnosis)
    [related_work p2] Soft computing techniques, such as Fuzzy sets and Fuzzy logic, Artificial Neural Networks (ANN), and Genetic Algorithms (GA) are u...
    [introduction p2] of symptoms there is a possibility of multiple diseases evident in case of depressi

## Step 5 — Connect Gemini

This step connects the RAG pipeline to Gemini by trying a list of available free-tier model candidates and using the first model that successfully responds.

This avoids depending on a single model name whose availability or quota may change during development.

In [ ]:
from google.colab import userdata
!pip -q install langchain-google-genai

import os
from langchain_google_genai import ChatGoogleGenerativeAI

os.environ["GOOGLE_API_KEY"] = userdata.get("Gemini_API")

CANDIDATES = ["gemini-flash-latest", "gemini-2.5-flash", "gemini-2.5-flash-lite",
              "gemini-flash-lite-latest", "gemini-3.1-flash-lite", "gemini-3.6-flash"]

llm = None
for name in CANDIDATES:
    try:
        test_llm = ChatGoogleGenerativeAI(model=name, temperature=0)
        test_llm.invoke("Say OK.")   # one tiny call to confirm it actually works, not just exists
        llm = test_llm
        print(f"✅ {name} works — using this model")
        break
    except Exception as e:
        print(f"❌ {name} — {type(e).__name__}: {str(e)[:100]}")

if llm is None:
    raise RuntimeError("None of the candidates worked — quota may be exhausted across the "
                        "board today, or your key needs a different model list.")


❌ gemini-flash-latest — GoogleRateLimitError: Error calling model 'gemini-flash-latest' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'
❌ gemini-2.5-flash — GoogleModelNotFoundError: Error calling model 'gemini-2.5-flash' (NOT_FOUND): 404 NOT_FOUND. {'error': {'code': 404, 'message'
❌ gemini-2.5-flash-lite — GoogleModelNotFoundError: Error calling model 'gemini-2.5-flash-lite' (NOT_FOUND): 404 NOT_FOUND. {'error': {'code': 404, 'mes
✅ gemini-flash-lite-latest works — using this model


## Step 6 — Shared Helpers

This step adds two utility functions used throughout the pipeline: `safe_invoke()` and `check_grounding()`.

`safe_invoke()` handles temporary `RESOURCE_EXHAUSTED` rate-limit errors, while `check_grounding()` verifies that the page numbers cited in an answer are actually present in the retrieved evidence.

In [ ]:
import re, time

def safe_invoke(chain, inputs, config=None, retries=3, wait=15):
    for attempt in range(retries):
        try:
            return chain.invoke(inputs, config=config) if config else chain.invoke(inputs)
        except Exception as e:
            msg = str(e)
            if "RESOURCE_EXHAUSTED" in msg and attempt < retries - 1:
                print(f"  rate limited, waiting {wait}s...")
                time.sleep(wait)
            elif "RESOURCE_EXHAUSTED" in msg:
                print("  still exhausted after retries — this looks like a DAILY cap, not transient. Waiting longer won't help.")
                raise
            else:
                raise

def check_grounding(answer, hits):
    cited_pages = set(int(p) for p in re.findall(r"\(?p(\d+)\)?", answer))
    retrieved_pages = set(h.metadata["page"] for h in hits)
    ungrounded = cited_pages - retrieved_pages
    if ungrounded:
        print(f"⚠️  GROUNDING FAILURE: answer cites page(s) {ungrounded}, "
              f"but retrieved chunks only came from pages {retrieved_pages}")
    else:
        print(f"✅ all cited pages ({cited_pages}) were actually retrieved ({retrieved_pages})")

## Step 7 — Single-Paper Grounded Answer Chain

This step creates the basic question-answering chain for a single research paper.

The chain uses only the retrieved excerpts to generate a page-cited answer and is instructed to explicitly state when the available evidence does not support an answer instead of guessing.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

single_paper_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are analyzing ONE research paper's excerpts. Answer ONLY using the excerpts given. "
     "If the excerpts don't clearly support an answer, say so explicitly instead of guessing. "
     "Cite the page number for every claim, like (p7)."),
    ("human",
     "Paper: {paper}\n\nExcerpts:\n{context}\n\nQuestion: {question}")
])
single_paper_chain = single_paper_prompt | llm | StrOutputParser()

# --- test on one paper, whole-paper search (no section gate) ---
paper = "Saha2024 (Fuzzy logic depression level)"
question = "What limitations or shortcomings does this paper mention?"
hits = per_paper_retrieve(question, k=3, papers=[paper])[paper]

context = "\n\n".join(f"(p{h.metadata['page']}) {h.page_content}" for h in hits)
answer = safe_invoke(single_paper_chain, {"paper": paper, "context": context, "question": question})

print("--- retrieved chunks used ---")
print(context)     # full context, no truncation — a short slice can hide real citations
print("\n--- Gemini's answer ---")
print(answer)
check_grounding(answer, hits)


--- retrieved chunks used ---
(p16) level could be a future project. In addition, further research, including a more varied sample representing many populations, could improve generalizability. This could involve expanding the sample size and including individuals from diverse and inclusive demographic backgrounds.

(p15) unavailable. The model’s ability to perform well on unseen data, as shown in Eq. (11), is a crucial indicator of its generalization capacity, meaning it can extend beyond the specific training set and offer reliable predictions in real-world scenarios. This is particularly important for applications in mental health, where individual variations in feelings and activities can differ greatly from person to person. Achieving a high level of accuracy across a diverse set of participants suggests that the fuzzy logic-based rules of the model are robust and adaptable to various patterns of human behavior. This study enhanced mental health predictions, bridging theory and pr

-------------------------------------
# CROSS-PAPER Q&A
-------------------------------------

## Step 8 — Cross-Paper Synthesis

This step extends the RAG pipeline to analyse all five research papers together using an LCEL map-reduce approach.

The map stage analyses each paper independently and produces grounded, page-cited evidence. The reduce stage then combines these results into a cross-paper synthesis showing relevant similarities and differences.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# --- map step: summarize ONE paper's evidence toward the question ---
map_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are extracting evidence from ONE paper's excerpts for a cross-paper research question. "
     "Summarize ONLY what these excerpts say, in 2-3 sentences, citing pages like (p7). "
     "If the excerpts don't address the question, say 'No relevant evidence found' — never guess."),
    ("human", "Paper: {paper}\n\nExcerpts:\n{context}\n\nQuestion: {question}")
])
map_chain = map_prompt | llm | StrOutputParser()

# --- reduce step: combine per-paper summaries into one synthesis ---
reduce_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are synthesizing evidence across MULTIPLE papers. You're given each paper's evidence summary "
     "(already grounded, with page citations). Identify which papers share a common point and where they "
     "differ. Do NOT introduce facts beyond what's in the summaries. Keep citations exactly as given."),
    ("human", "Question: {question}\n\nPer-paper evidence:\n{summaries}")
])
reduce_chain = reduce_prompt | llm | StrOutputParser()

def _map_evidence(question, k=3, sections=None, exclude_table_like=True):
    """Shared step: retrieve top-k chunks per paper, then summarize each paper's evidence.
    Used by both cross_paper_synthesize (free-text) and analyze_gap (structured)."""
    per_paper = per_paper_retrieve(question, k=k, sections=sections, exclude_table_like=exclude_table_like)
    summaries, all_hits = [], []
    for paper, hits in per_paper.items():
        if not hits:
            summaries.append(f"{paper}: No chunks retrieved.")
            continue
        context = "\n\n".join(f"(p{h.metadata['page']}) {h.page_content}" for h in hits)
        summary = safe_invoke(map_chain, {"paper": paper, "context": context, "question": question})
        summaries.append(f"{paper}: {summary}")
        all_hits.extend(hits)
    return "\n\n".join(summaries), all_hits

def cross_paper_synthesize(question, k=3, sections=None, exclude_table_like=True):
    combined, all_hits = _map_evidence(question, k=k, sections=sections, exclude_table_like=exclude_table_like)
    final = safe_invoke(reduce_chain, {"question": question, "summaries": combined})

    print("--- per-paper summaries (map step) ---")
    print(combined)
    print("\n--- cross-paper synthesis (reduce step) ---")
    print(final)
    check_grounding(final, all_hits)
    return final, all_hits

# --- one test, spending 6 calls, whole-paper search this time ---
final, hits = cross_paper_synthesize("What limitations are repeatedly mentioned across the papers?")


--- per-paper summaries (map step) ---
Adegboye2021 (Genetic Neuro-Fuzzy): No relevant evidence found

Chattopadhyay2017 (Neuro-fuzzy diagnosis): No relevant evidence found

Khan2024 (EEG temporal features + ML): No relevant evidence found

Saha2024 (Fuzzy logic depression level): No relevant evidence found.

Zulfiker2021 (ML + feature selection): No relevant evidence found

--- cross-paper synthesis (reduce step) ---
Based on the provided evidence summaries, none of the papers contain information regarding limitations (each paper states "No relevant evidence found"). Therefore, no shared or differing limitations can be identified across the papers based on the provided text.
✅ all cited pages (set()) were actually retrieved ({1, 2, 3, 4, 5, 6, 7, 9, 10, 16, 23})


## Step 9 — Pydantic Structured Output

This step introduces structured output for research gaps using a `ResearchGap` schema containing the identified gap, related papers, and supporting evidence.

`PydanticOutputParser` is used to parse the model's response into this structured format. `analyze_gap()` reuses the `_map_evidence` step from Step 8 and produces structured research gaps directly, reducing the pipeline from 7 model calls to 6.

The prompt also uses stricter merge rules: two limitations are combined only when they share the same underlying cause, rather than simply having a common downstream consequence.

In [ ]:
from typing import List
from pydantic import BaseModel, Field
from langchain_core.output_parsers import PydanticOutputParser

# --- 1. the schema ---
class Evidence(BaseModel):
    paper: str = Field(description="Short paper label, e.g. 'Saha2024 (Fuzzy logic depression level)'")
    page: int = Field(description="Page number this evidence came from")
    quote_or_paraphrase: str = Field(description="A short quote or close paraphrase supporting the gap")

class ResearchGap(BaseModel):
    gap: str = Field(description="A concise statement of the limitation, unresolved problem, or gap")
    papers: List[str] = Field(description="Short labels of every paper that mentions this gap")
    evidence: List[Evidence] = Field(description="Supporting evidence, one entry per paper/quote")

class ResearchGapList(BaseModel):
    gaps: List[ResearchGap] = Field(description="All distinct gaps identified across the papers")

parser = PydanticOutputParser(pydantic_object=ResearchGapList)

# --- 2. structured synthesis prompt (stricter merge rule) ---
structured_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are extracting structured research gaps from per-paper evidence summaries. "
     "Each gap must be traceable to specific pages already present in the summaries — "
     "never invent a page number or paper name that isn't in the input. "
     "Only merge two papers into the SAME gap if they describe the same underlying cause or "
     "mechanism (e.g. both restrict their sample/dataset in the same way). Do NOT merge two "
     "papers just because their limitations share a downstream consequence like 'generalizability' "
     "— 'small sample diversity' and 'single-lobe EEG scope' are DIFFERENT gaps even though both "
     "affect generalizability. When in doubt, keep gaps separate rather than merge them. "
     "{format_instructions}"),
    ("human", "Question: {question}\n\nPer-paper evidence:\n{summaries}")
])

structured_chain = structured_prompt.partial(
    format_instructions=parser.get_format_instructions()
) | llm | parser

def analyze_gap(question, k=3):
    """End-to-end structured pipeline: retrieve -> per-paper map summaries -> Pydantic gaps.
    Skips the prose reduce step from Step 8 (not needed for structured output, and cheaper)."""
    combined, all_hits = _map_evidence(question, k=k)
    result = safe_invoke(structured_chain, {"question": question, "summaries": combined})
    for g in result.gaps:
        print(f"GAP: {g.gap}")
        print(f"  Papers: {g.papers}")
        for e in g.evidence:
            print(f"  - [{e.paper} p{e.page}] {e.quote_or_paraphrase}")
        print()
    return result, all_hits

# --- one live test, ~6 calls ---
result, hits = analyze_gap("What limitations are repeatedly mentioned?")


## Step 10 — Conversation Memory

This step adds conversation memory using `RunnableWithMessageHistory`, allowing follow-up questions to refer to information from the previous interaction.

The memory is stored temporarily in memory and resets when the Colab runtime restarts. It helps the generation step resolve references such as "it" or "those", but retrieval itself is not yet history-aware.

In [ ]:
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.prompts import MessagesPlaceholder

# --- 1. memory-aware prompt: history is for resolving "it"/"those", NOT a source of facts ---
memory_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are analyzing research paper excerpts. Answer ONLY using the excerpts given for THIS turn. "
     "Use the conversation history only to understand what the user is referring to (e.g. 'that paper', "
     "'those limitations') — never treat prior answers as a source of new facts. "
     "Cite page numbers like (p7). If the excerpts don't support an answer, say so."),
    MessagesPlaceholder("history"),
    ("human", "Paper: {paper}\n\nExcerpts:\n{context}\n\nQuestion: {question}")
])
memory_chain = memory_prompt | llm | StrOutputParser()

# --- 2. in-memory session store (resets when the Colab runtime resets — fine for a demo) ---
store = {}
def get_history(session_id):
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]

chain_with_memory = RunnableWithMessageHistory(
    memory_chain, get_history,
    input_messages_key="question", history_messages_key="history",
)

def ask_with_memory(paper, question, session_id="demo", k=3, sections=None, exclude_table_like=True):
    hits = per_paper_retrieve(question, k=k, sections=sections, exclude_table_like=exclude_table_like,
                               papers=[paper])[paper]
    context = "\n\n".join(f"(p{h.metadata['page']}) {h.page_content}" for h in hits)
    config = {"configurable": {"session_id": session_id}}
    answer = safe_invoke(chain_with_memory, {"paper": paper, "context": context, "question": question},
                          config=config)
    print(f"Q: {question}\nA: {answer}\n")
    return answer, hits

# --- test: 2 turns, second depends on the first (costs 2 API calls) ---
paper = "Saha2024 (Fuzzy logic depression level)"
ask_with_memory(paper, "What limitations does this paper mention?")
ask_with_memory(paper, "Of those, which seems easiest for future researchers to actually fix?")


/usr/local/lib/python3.13/dist-packages/IPython/core/interactiveshell.py:3553: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


Q: What limitations does this paper mention?
A: Based on the provided excerpts, the paper mentions the following limitations and areas for improvement regarding generalizability:

* The model's generalizability could be improved by including a more varied sample representing many populations (p15, p16).
* This improvement could involve expanding the sample size and including individuals from diverse and inclusive demographic backgrounds (p15, p16).

Q: Of those, which seems easiest for future researchers to actually fix?
A: Based on the provided excerpts, there is no mention or comparison of which limitation is the easiest for future researchers to fix. Therefore, I cannot answer this using the given text.



('Based on the provided excerpts, there is no mention or comparison of which limitation is the easiest for future researchers to fix. Therefore, I cannot answer this using the given text.',
 [Document(id='3b87fdd4-9c8e-4927-a5e6-9f717bc2881a', metadata={'paper': 'Saha2024 (Fuzzy logic depression level)', 'section': 'discussion', 'page': 16, 'table_like': False, 'chunk_id': 266}, page_content='could also use such predictive tools for early interventions, integrating them into support services. • Policymakers could leverage such predictive data to advocate for mental health funding, as the model’s insights can highlight the prevalence and risk factors associated with depression. • Additionally, such fuzzy logic-based solutions can also be applied to other domains, such as cybersecurity Sarker (2023). As an example, ambiguous Internet traffic patterns can be evaluated to detect unusual or suspicious network activity.'),
  Document(id='2d566c7d-db75-4eb7-9c49-cf176f09a9c7', metadata={'pape

## Step 11 — Formal evaluation

This step runs the formal evaluation set of six questions covering different research-gap cases, including an accuracy conflict and a PCA-related retrieval case.

The results are saved progressively to `eval/results.json`, allowing completed questions to be preserved and the evaluation to be resumed if a quota error interrupts the run.

The evaluation process uses a resumable evaluation harness with six questions: four gap-related questions, one accuracy-conflict case, and one PCA-count case. Results are saved progressively to eval/results.json, ensuring that previously completed responses are not lost if a quota error (429) occurs during execution. When the evaluation cell is run again, it automatically skips questions that have already been completed.

Since cross_paper_synthesize now searches the full papers by default following the Step 4/8 fix, this evaluation serves as a genuine before-and-after comparison with the previous evaluation run. Particular attention should be given to:

- conflict_accuracy — whether the system now identifies the conflicting accuracy values reported by Chattopadhyay (Table 7: 92.03%) and Adegboye (96.9%).
- failure_pca_count — whether the system can now identify Chattopadhyay's use of PCA in the Methods section, rather than incorrectly reporting zero.

Regardless of whether the results improve or remain unchanged, both outcomes are valid and should be reported objectively. The evaluation should not be rerun repeatedly in an attempt to obtain a more favorable result.

In [ ]:
import json, time, os

os.makedirs("eval", exist_ok=True)
RESULTS_PATH = "eval/results.json"

TEST_QUESTIONS = [
    ("gap_limitations",  "What limitations are repeatedly mentioned across the papers?"),
    ("gap_comparisons",  "Which approaches or methods are compared across the papers?"),
    ("gap_unresolved",   "What problems remain unresolved across the papers?"),
    ("gap_future_work",  "What future work do the authors propose across the papers?"),
    ("conflict_accuracy","Which paper reports the highest accuracy?"),
    ("failure_pca_count","How many papers use PCA?"),
]

# resume support: reload whatever's already been saved, skip those questions
try:
    all_results = json.load(open(RESULTS_PATH))
except FileNotFoundError:
    all_results = {}

for key, question in TEST_QUESTIONS:
    if key in all_results:
        print(f"skipping '{key}' — already saved from a previous run")
        continue
    print(f"\n{'='*100}\n{key}: {question}")
    try:
        final, hits = cross_paper_synthesize(question)
        all_results[key] = {
            "question": question,
            "synthesis": final,
            "retrieved": [{"paper": h.metadata["paper"], "section": h.metadata["section"],
                           "page": h.metadata["page"], "text": h.page_content} for h in hits],
        }
        json.dump(all_results, open(RESULTS_PATH, "w"), indent=1)
        print("saved. pausing 20s before next question...")
        time.sleep(20)
    except Exception as e:
        print(f"\n⚠️ stopped at '{key}': {type(e).__name__}: {str(e)[:200]}")
        print("Progress is saved in eval/results.json — just rerun this same cell later, it'll skip what's done.")
        break

print(f"\ndone: {len(all_results)}/{len(TEST_QUESTIONS)} questions completed")



gap_limitations: What limitations are repeatedly mentioned across the papers?
--- per-paper summaries (map step) ---
Adegboye2021 (Genetic Neuro-Fuzzy): No relevant evidence found

Chattopadhyay2017 (Neuro-fuzzy diagnosis): No relevant evidence found

Khan2024 (EEG temporal features + ML): No relevant evidence found

Saha2024 (Fuzzy logic depression level): No relevant evidence found

Zulfiker2021 (ML + feature selection): No relevant evidence found.

--- cross-paper synthesis (reduce step) ---
Based on the provided evidence summaries, none of the papers contain information regarding limitations. Therefore, no shared or differing limitations can be identified across the papers (Adegboye2021, Chattopadhyay2017, Khan2024, Saha2024, and Zulfiker2021 all report "No relevant evidence found").
✅ all cited pages (set()) were actually retrieved ({1, 2, 3, 4, 5, 6, 7, 9, 10, 16, 23})
saved. pausing 20s before next question...

gap_comparisons: Which approaches or methods are compared across th

Save Progress

In [ ]:
!zip -r /content/ResearchGapAnalyzer_backup.zip \
    /content/papers \
    /content/papers_txt \
    /content/faiss_index \
    /content/eval \
    /content/chunks.json

  adding: content/papers/ (stored 0%)
  adding: content/papers/A_neuro-fuzzy_approach.pdf (deflated 24%)
  adding: content/papers/Predicting_depression_level_based.pdf (deflated 7%)
  adding: content/papers/AGeneticNeuro-FuzzySystemforDiagnosingClinical.pdf (deflated 19%)
  adding: content/papers/An_in-depth_analysis_of_machine.pdf (deflated 31%)
  adding: content/papers/A_machine_learning_based_depression_screening_fram.pdf (deflated 24%)
  adding: content/papers_txt/ (stored 0%)
  adding: content/papers_txt/A_neuro-fuzzy_approach.pdf.txt (deflated 61%)
  adding: content/papers_txt/A_machine_learning_based_depression_screening_fram.pdf.txt (deflated 66%)
  adding: content/papers_txt/AGeneticNeuro-FuzzySystemforDiagnosingClinical.pdf.txt (deflated 62%)
  adding: content/papers_txt/An_in-depth_analysis_of_machine.pdf.txt (deflated 66%)
  adding: content/papers_txt/Predicting_depression_level_based.pdf.txt (deflated 64%)
  adding: content/faiss_index/ (stored 0%)
  adding: content/faiss_

In [ ]:
from google.colab import files
files.download('/content/ResearchGapAnalyzer_backup.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>